In [1]:
# ============================================================
# CONFIGURAÇÃO INICIAL DO AMBIENTE
# ============================================================
#
# Nesta etapa, preparamos o ambiente para que o Spark consiga interagir com o S3 
# utilizando as credenciais temporárias do Learner Lab.
#
# Pontos de configuração esperados no .env:
#
#   AWS_ACCESS_KEY_ID      Chave de acesso temporária da AWS.
#   AWS_SECRET_ACCESS_KEY  Chave secreta temporária da AWS.
#   AWS_SESSION_TOKEN      Token temporário da sessão AWS Academy.
#   S3_BUCKET              Nome do bucket S3 do integrante.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# Setup Env: Dotenv/Pathlib
import os, sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# ============================================================
# CAMINHOS DO PROJETO E VARIÁVEIS DE AMBIENTE
# ============================================================

# Localiza o arquivo .env, carrega suas variáveis e define a raiz do projeto.
load_dotenv(find_dotenv())
PROJECT_ROOT = Path(find_dotenv()).parent

# ============================================================
# CONFIGURAÇÃO DO PYSPARK
# ============================================================

# Garante que o PySpark use o mesmo interpretador Python do ambiente atual.
# Isso evita conflitos comuns em WSL, Conda ou ambientes virtuais.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ============================================================
# CREDENCIAIS AWS E BUCKET S3
# ============================================================

# Carrega as credenciais temporárias e o bucket S3 definidos no .env.
AWS_ACCESS_KEY_ID     = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN     = os.getenv("AWS_SESSION_TOKEN")
S3_BUCKET             = os.getenv("S3_BUCKET")

# Interrompe a execução se alguma configuração obrigatória não foi carregada.
if not all([AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN, S3_BUCKET]):
    raise EnvironmentError("Falha ao carregar credenciais AWS ou S3_BUCKET do arquivo .env")

# ============================================================
# CONFERÊNCIA DA CONFIGURAÇÃO
# ============================================================

print(f"Projeto: {PROJECT_ROOT}\nBucket : {S3_BUCKET}")

Projeto: /mnt/d/diego/01_projects/postech-challenge-2
Bucket : alfabetizacao-data-lake-diego


In [2]:
# ============================================================
# CRIAÇÃO DA SESSÃO SPARK COM ACESSO AO S3
# ============================================================
#
# Esta célula cria a SparkSession usada para gravar os resultados da camada Silver no S3.
#
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# SparkSession: S3 Config
from pyspark.sql import SparkSession

# ============================================================
# SESSÃO SPARK
# ============================================================

spark = (
    SparkSession.builder
    .appName("transformar-e-armazenar-S3")
    .master("local[*]") # remover quando for implementar a computação em nuvem
    # Configuração de Pacotes e S3A
    .config("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,""com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.session.token", AWS_SESSION_TOKEN)
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .getOrCreate()
)

# ============================================================
# CONFERÊNCIA DA SESSÃO
# ============================================================

print(f"Sessão Spark {spark.version} criada com sucesso.")

26/07/08 21:06:13 WARN Utils: Your hostname, lua resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/08 21:06:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/diego/miniconda3/envs/postech2/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/diego/.ivy2/cache
The jars for the packages stored in: /home/diego/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f0434b23-f845-4362-a16d-6e4cea3e0ac8;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 489ms :: artifacts dl 35ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evict

Sessão Spark 3.5.5 criada com sucesso.


In [3]:
# ============================================================
# CLIENTE S3 PARA OPERAÇÕES ADMINISTRATIVAS
# ============================================================
#
# Esta célula cria um cliente S3 com boto3 usando as mesmas
# credenciais temporárias carregadas do arquivo .env.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# Boto3: Gestão de pastas
import boto3

# ============================================================
# CLIENTE S3
# ============================================================

# Inicializa o cliente S3 logo após o setup das credenciais.
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

In [16]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

BRONZE_BASE = f"s3a://{S3_BUCKET}/bronze"
SILVER_BASE = f"s3a://{S3_BUCKET}/silver"
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

# Timestamp único da execução Silver (determinístico p/ todo o batch, via F.lit)
SILVER_TS = datetime.now(timezone.utc)

# --- Trechos de SQL reutilizados pelas 3 tabelas de META ---
def norm_pct(col):
    # Normaliza percentual das PLANILHAS DE META p/ DOUBLE.
    # '>80'/'> 80 ' -> 80.0  |  '- '/vazio/não-numérico -> NULL  |  número (ponto) -> double
    # Vírgula decimal: NÃO observada em 2023-2025 nos arquivos INEP -> ramo omitido de propósito.
    return (
        f"CASE "
        f"WHEN {col} IS NULL THEN NULL "
        f"WHEN TRIM({col}) LIKE '>%' THEN 80.0 "
        f"WHEN TRIM({col}) RLIKE '^[0-9]+([.][0-9]+)?$' THEN CAST(TRIM({col}) AS DOUBLE) "
        f"ELSE NULL END"
    )
    
# Diagonal: taxa do ano corrente, cada linha da SUA publicação.
DIAGONAL_TAXA = f"""
    CASE NU_ANO_AVALIACAO
        WHEN 2023 THEN {norm_pct('PC_ALUNO_ALFABETIZADO')}
        WHEN 2024 THEN {norm_pct('PC_ALUNO_ALFABETIZADO_2024')}
        WHEN 2025 THEN {norm_pct('PC_ALUNO_ALFABETIZADO_2025')}
    END AS taxa_alfabetizacao
"""

METAS_NORMALIZADAS = ",\n        ".join(
    f"{norm_pct(f'META_FINAL_{a}')} AS meta_alfabetizacao_{a}" for a in range(2024, 2031)
)

# Metadados de linhagem — adicionados de forma UNIFORME a toda tabela Silver.
# _silver_processed_at: carimbo único do batch (alinha com o modelo etl-silver.py)
def add_metadata(df):
    return df.withColumn(
        "_silver_processed_at",
        F.lit(SILVER_TS)
    )

## Tabelas:
- UF;
- Meta Alfabetização Brasil;
- Meta Alfabetização por UF;
- Meta Alfabetização por Município;
- Município;
- Dados de aluno.

### Processamento da Meta Nacional (Brasil)
Nesta etapa, filtramos os dados agregados ao nível federal para os anos de interesse (2023-2025). O objetivo é consolidar os indicadores de alfabetização e as metas normalizadas na camada Silver, garantindo que os dados nacionais estejam isolados, enriquecidos com metadados de rastreabilidade e prontos para análise comparativa.

In [17]:
metas_uf_bronze = spark.read.option("mergeSchema", "true").parquet(f"{BRONZE_BASE}/metas_ufs")
metas_uf_bronze.createOrReplaceTempView("bronze_metas_ufs")

brasil_silver = add_metadata(spark.sql(f"""
    SELECT
        NU_ANO_AVALIACAO AS ano,
        INITCAP(REDE)    AS rede,
        {DIAGONAL_TAXA},
        {METAS_NORMALIZADAS},
        PC_AVALIADOS_LP  AS percentual_participacao
    FROM bronze_metas_ufs
    WHERE NOME_UF = 'Brasil'
      AND NU_ANO_AVALIACAO IN (2023, 2024, 2025)
"""))

(brasil_silver.write.mode("overwrite").partitionBy("ano")
    .parquet(f"{SILVER_BASE}/meta_alfabetizacao_brasil"))

print("meta_alfabetizacao_brasil gravada.")
brasil_silver.orderBy(F.col("ano").desc()).show(truncate=False)

meta_alfabetizacao_brasil gravada.


+----+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+--------------------------+
|ano |rede   |taxa_alfabetizacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|percentual_participacao|_silver_processed_at      |
+----+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+--------------------------+
|2025|Pública|66.0              |60.0                   |64.0                   |67.0                   |71.0                   |74.0                   |77.0                   |80.0                   |89.0                   |2026-07-08 17:33

### Processamento das Metas por Unidade da Federação (UF)
Nesta etapa, realizamos a filtragem e o refinamento dos dados das metas de alfabetização em nível estadual. Diferente da célula anterior, aqui excluímos o agregado nacional ('Brasil') para focar exclusivamente nas UFs, garantindo a padronização dos nomes das redes de ensino e a tipagem correta dos indicadores para os anos de 2023 a 2025. Os dados resultantes são persistidos na camada Silver, particionados por ano para otimizar a performance de consultas geográficas e temporais.

In [18]:
metas_uf_bronze = spark.read.option("mergeSchema", "true").parquet(f"{BRONZE_BASE}/metas_ufs")
metas_uf_bronze.createOrReplaceTempView("bronze_metas_ufs")

meta_uf_silver = add_metadata(spark.sql(f"""
    SELECT
        NU_ANO_AVALIACAO AS ano,
        SIGLA_UF         AS sigla_uf,
        INITCAP(REDE)    AS rede,
        {DIAGONAL_TAXA},
        {METAS_NORMALIZADAS},
        PC_AVALIADOS_LP  AS percentual_participacao
    FROM bronze_metas_ufs
    WHERE NOME_UF IS NOT NULL
      AND NOME_UF <> 'Brasil'
      AND NU_ANO_AVALIACAO IN (2023, 2024, 2025)
"""))

(meta_uf_silver.write.mode("overwrite").partitionBy("ano")
    .parquet(f"{SILVER_BASE}/meta_alfabetizacao_uf"))
print("meta_alfabetizacao_uf gravada. Total:", meta_uf_silver.count())
meta_uf_silver.orderBy("ano", "sigla_uf").show(5, truncate=False)

26/07/08 17:34:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/08 17:34:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/07/08 17:34:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/07/08 17:34:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/07/08 17:34:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


meta_alfabetizacao_uf gravada. Total: 81


+----+--------+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+--------------------------+
|ano |sigla_uf|rede   |taxa_alfabetizacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|percentual_participacao|_silver_processed_at      |
+----+--------+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+--------------------------+
|2023|AC      |Pública|NULL              |NULL                   |NULL                   |NULL                   |NULL                   |NULL                   |NULL                   |NULL                   |NULL

### Processamento das Metas Municipais
Nesta etapa, realizamos o refinamento dos dados de metas de alfabetização para o nível municipal. A lógica de transformação inclui a padronização do identificador do município, a formatação da rede de ensino e a consolidação dos níveis de alfabetização — utilizando a função COALESCE para garantir a compatibilidade entre os diferentes campos de nível técnico de 2023 e anos posteriores. Os dados são filtrados para o período de 2023 a 2025 e persistidos na camada Silver com particionamento por ano para otimizar a performance de consultas analíticas.

In [19]:
metas_mun_bronze = spark.read.option("mergeSchema", "true").parquet(f"{BRONZE_BASE}/metas_municipios")
metas_mun_bronze.createOrReplaceTempView("bronze_metas_municipios") #view temporária

meta_mun_silver = add_metadata(spark.sql(f"""
    SELECT
        NU_ANO_AVALIACAO AS ano,
        CO_MUNICIPIO     AS id_municipio,
        INITCAP(NO_TP_REDE) AS rede,
        {DIAGONAL_TAXA},
        {METAS_NORMALIZADAS},
        COALESCE(NIVEIS_ALFABETIZACAO_2023, CO_NIVEL_ALFABETIZACAO) AS nivel_alfabetizacao,
        PC_AVALIADOS_LP  AS percentual_participacao
    FROM bronze_metas_municipios
    WHERE CO_MUNICIPIO IS NOT NULL
      AND NU_ANO_AVALIACAO IN (2023, 2024, 2025)
"""))

(meta_mun_silver.write.mode("overwrite").partitionBy("ano")
    .parquet(f"{SILVER_BASE}/meta_alfabetizacao_municipio"))
print("meta_alfabetizacao_municipio gravada. Total:", meta_mun_silver.count())
meta_mun_silver.orderBy("ano", "id_municipio").show(5, truncate=False)


26/07/08 17:36:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/08 17:36:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/07/08 17:36:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/07/08 17:36:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/07/08 17:36:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/07/08 17:36:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/07/08 17:36:04 WARN MemoryManager: Total allocation exceeds 95.

meta_alfabetizacao_municipio gravada. Total: 16286


+----+------------+---------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------+-----------------------+--------------------------+
|ano |id_municipio|rede     |taxa_alfabetizacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|nivel_alfabetizacao|percentual_participacao|_silver_processed_at      |
+----+------------+---------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------+-----------------------+--------------------------+
|2023|1100015     |Municipal|64.6              |67.08                  |69.51                  |71.84                  |74.06           

### Processamento dos Dados Históricos Estaduais (UF)
Nesta etapa, realizamos o refinamento dos dados de séries históricas em nível estadual. O processo envolve a aplicação de tipagem explícita (Casting) para garantir a integridade do esquema (schema) na camada Silver, convertendo identificadores, anos e indicadores de desempenho para formatos numéricos adequados. A filtragem garante que apenas registros com identificação válida de UF sejam processados. Os dados finais são enriquecidos com metadados e persistidos no S3, utilizando o particionamento por ano para facilitar análises temporais de longo prazo.

In [20]:
uf_bronze = spark.read.option("mergeSchema", "true").parquet(f"{BRONZE_BASE}/ts_estado")
uf_bronze.createOrReplaceTempView("bronze_ts_estado")

uf_silver = add_metadata(spark.sql("""
    SELECT
        CAST(NU_ANO_AVALIACAO AS INT)          AS ano,
        CAST(CO_UF AS INT)                     AS id_uf,
        SG_UF                                  AS sigla_uf,
        CAST(TP_SERIE AS INT)                  AS serie,
        CAST(ID_TIPO_REDE AS INT)              AS rede,
        CAST(PC_ALUNO_ALFABETIZADO AS DOUBLE)  AS taxa_alfabetizacao,
        CAST(VL_MEDIA_LP AS DOUBLE)            AS media_portugues
    FROM bronze_ts_estado
    WHERE CO_UF IS NOT NULL
"""))

(uf_silver.write.mode("overwrite").partitionBy("ano")
    .parquet(f"{SILVER_BASE}/uf"))
print("silver/uf gravada. Total:", uf_silver.count())
uf_silver.orderBy("ano", "id_uf", "rede").show(10, truncate=False)

silver/uf gravada. Total: 225


+----+-----+--------+-----+----+------------------+---------------+--------------------------+
|ano |id_uf|sigla_uf|serie|rede|taxa_alfabetizacao|media_portugues|_silver_processed_at      |
+----+-----+--------+-----+----+------------------+---------------+--------------------------+
|2023|11   |RO      |2    |2   |58.65             |751.4731       |2026-07-08 17:33:37.114728|
|2023|11   |RO      |2    |3   |65.17             |760.1971       |2026-07-08 17:33:37.114728|
|2023|11   |RO      |2    |5   |64.6              |759.4357       |2026-07-08 17:33:37.114728|
|2023|13   |AM      |2    |2   |62.63             |746.2058       |2026-07-08 17:33:37.114728|
|2023|13   |AM      |2    |3   |49.2              |733.6637       |2026-07-08 17:33:37.114728|
|2023|13   |AM      |2    |5   |52.2              |736.4687       |2026-07-08 17:33:37.114728|
|2023|15   |PA      |2    |2   |57.07             |741.3548       |2026-07-08 17:33:37.114728|
|2023|15   |PA      |2    |3   |47.71             

### Processamento dos Dados Históricos Municipais
Nesta etapa, realizamos o refinamento e a padronização dos dados históricos em nível municipal. Um ponto crítico desta transformação é o tratamento do código do município (CO_MUNICIPIO), onde aplicamos uma combinação de CAST, TRIM e LPAD para garantir que o identificador seja uma string de 7 dígitos (padrão IBGE), prevenindo a perda de zeros à esquerda. Além disso, aplicamos tipagem numérica rigorosa aos indicadores de alfabetização e média de português, garantindo a consistência para análises comparativas. Os dados são persistidos na camada Silver, particionados por ano.

In [21]:
mun_bronze = spark.read.option("mergeSchema", "true").parquet(f"{BRONZE_BASE}/ts_municipio")
mun_bronze.createOrReplaceTempView("bronze_ts_municipio")

municipio_silver = add_metadata(spark.sql("""
    SELECT
        CAST(NU_ANO_AVALIACAO AS INT)                     AS ano,
        LPAD(TRIM(CAST(CO_MUNICIPIO AS STRING)), 7, '0')  AS id_municipio,
        TRIM(NO_MUNICIPIO)                                AS nome_municipio,
        CAST(CO_UF AS INT)                                AS id_uf,
        SG_UF                                             AS sigla_uf,
        CAST(TP_SERIE AS INT)                             AS serie,
        CAST(ID_TIPO_REDE AS INT)                         AS rede,
        CAST(PC_ALUNO_ALFABETIZADO AS DOUBLE)             AS taxa_alfabetizacao,
        CAST(VL_MEDIA_LP AS DOUBLE)                        AS media_portugues
    FROM bronze_ts_municipio
    WHERE CO_MUNICIPIO IS NOT NULL
"""))

(municipio_silver.write.mode("overwrite").partitionBy("ano")
    .parquet(f"{SILVER_BASE}/municipio"))
print("silver/municipio gravada. Total:", municipio_silver.count())
municipio_silver.orderBy("ano", "id_municipio", "rede").show(10, truncate=False)

silver/municipio gravada. Total: 36411


+----+------------+---------------------+-----+--------+-----+----+------------------+---------------+--------------------------+
|ano |id_municipio|nome_municipio       |id_uf|sigla_uf|serie|rede|taxa_alfabetizacao|media_portugues|_silver_processed_at      |
+----+------------+---------------------+-----+--------+-----+----+------------------+---------------+--------------------------+
|2023|1100015     |Alta Floresta D'Oeste|11   |RO      |2    |3   |64.55             |758.3304       |2026-07-08 17:33:37.114728|
|2023|1100015     |Alta Floresta D'Oeste|11   |RO      |2    |5   |64.55             |758.3304       |2026-07-08 17:33:37.114728|
|2023|1100023     |Ariquemes            |11   |RO      |2    |3   |62.3              |757.0999       |2026-07-08 17:33:37.114728|
|2023|1100023     |Ariquemes            |11   |RO      |2    |5   |62.3              |757.0999       |2026-07-08 17:33:37.114728|
|2023|1100031     |Cabixi               |11   |RO      |2    |3   |69.1              |767.

### Processamento dos Microdados de Alunos
Nesta etapa, realizamos o refinamento dos dados individuais dos alunos. Esta é a célula mais crítica em termos de volume e regras de negócio. Aplicamos a padronização do código do município (7 dígitos) e a limpeza de identificadores de escola e aluno.

Regra de Qualidade Importante: Foi implementada uma lógica via CASE WHEN para o campo alfabetizado. Registros onde a proficiência é nula (estudantes não medidos) são mantidos como NULL em vez de zero, protegendo o denominador em cálculos futuros de taxas e médias. Os dados são enriquecidos com metadados e persistidos na camada Silver, particionados por ano para suportar a volumetria dos microdados.

In [22]:
alunos_bronze = spark.read.option("mergeSchema", "true").parquet(f"{BRONZE_BASE}/ts_aluno")
alunos_bronze.createOrReplaceTempView("bronze_ts_aluno")

alunos_silver = add_metadata(spark.sql("""
    WITH base AS (
        SELECT *, CAST(VL_PROFICIENCIA_LP AS DOUBLE) AS _prof
        FROM bronze_ts_aluno
        WHERE ID_ALUNO IS NOT NULL
    )
    SELECT
        CAST(NU_ANO_AVALIACAO AS INT)                     AS ano,
        LPAD(TRIM(CAST(CO_MUNICIPIO AS STRING)), 7, '0')  AS id_municipio,
        TRIM(CAST(ID_ESCOLA AS STRING))                   AS id_escola,
        TRIM(CAST(ID_ALUNO  AS STRING))                   AS id_aluno,
        CAST(CO_CADERNO_LP AS INT)                        AS caderno,
        CAST(TP_SERIE AS INT)                             AS serie,
        CAST(TP_DEPENDENCIA AS INT)                       AS rede,
        CAST(IN_PRESENCA_LP AS INT)                       AS presenca,
        CAST(IN_PREENCHIMENTO_LP AS INT)                  AS preenchimento_caderno,
        CASE WHEN _prof IS NULL THEN NULL                 -- 769.525 não-medidos -> NULL protege o denominador
             ELSE CAST(IN_ALFABETIZADO AS INT) END        AS alfabetizado,
        _prof                                             AS proficiencia,
        CAST(VL_PESO_ALUNO_LP AS DOUBLE)                  AS peso_aluno
    FROM base
"""))

(alunos_silver.write.mode("overwrite").partitionBy("ano")
    .parquet(f"{SILVER_BASE}/alunos"))
print("silver/alunos gravada. Total:", alunos_silver.count())
alunos_silver.show(5, truncate=False)

26/07/08 17:39:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/08 17:39:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/07/08 17:39:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/07/08 17:39:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/07/08 17:39:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/07/08 17:39:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/07/08 17:39:20 WARN MemoryManager: Total allocation exceeds 95.

silver/alunos gravada. Total: 6090791


+----+------------+---------+--------+-------+-----+----+--------+---------------------+------------+------------+----------+--------------------------+
|ano |id_municipio|id_escola|id_aluno|caderno|serie|rede|presenca|preenchimento_caderno|alfabetizado|proficiencia|peso_aluno|_silver_processed_at      |
+----+------------+---------+--------+-------+-----+----+--------+---------------------+------------+------------+----------+--------------------------+
|2025|3171204     |60019155 |31187261|6      |2    |3   |1       |1                    |0           |661.9209099 |1.1       |2026-07-08 17:33:37.114728|
|2025|3304300     |60024876 |33012647|3      |2    |3   |1       |1                    |1           |752.361904  |1.1538462 |2026-07-08 17:33:37.114728|
|2025|3171204     |60019155 |31187262|7      |2    |3   |1       |1                    |0           |680.3427531 |1.1       |2026-07-08 17:33:37.114728|
|2025|3304300     |60024876 |33012648|4      |2    |3   |0       |0               

### Validação de Qualidade (Data Quality) na Camada Silver
Esta etapa é fundamental para garantir a integridade dos dados antes que eles sejam consumidos por dashboards ou modelos analíticos. O objetivo é implementar um catálogo declarativo de verificações (CHECKS) que valida a saúde dos dados em quatro dimensões:

Contagem mínima (min_count): Garante que as tabelas não estejam vazias após o processamento.
Não-nulidade (not_null): Verifica colunas essenciais (chaves e anos) que não podem conter nulos.
Unicidade (unique): Valida chaves primárias ou compostas para evitar a duplicidade de registros.
Intervalo (range): Garante que indicadores de taxa e proficiência estejam dentro de limites estatísticos aceitáveis.
A função checar_qualidade atribui severidade às regras; caso um check definido como crítico falhe, a pipeline é interrompida imediatamente para evitar a propagação de dados corrompidos para as camadas subsequentes.

In [23]:
# ============================================================
# VALIDAÇÃO DE QUALIDADE — CAMADA SILVER
# ============================================================
# Espelha etl-silver.py: catálogo declarativo (CHECKS) + severidade
# por regra (critico) -> PASS/FAIL/WARN, Score e raise em falha crítica.
# Tipos: min_count, not_null, unique (aceita chave composta), range.
# (No .py a saída vai por logging; aqui usamos print p/ mostrar inline.)

def checar_qualidade(entidade, df, checks):
    print(f"[DQ:SILVER] {entidade} | iniciando | checks={len(checks)}")
    passou = falhou = criticos = 0

    for check in checks:
        tipo    = check["tipo"]
        coluna  = check.get("coluna")
        valor   = check.get("valor")
        critico = check.get("critico", True)
        ok, detalhe = False, ""

        try:
            if tipo == "min_count":
                n  = df.count()
                ok, detalhe = n >= valor, f"contagem={n} | minimo={valor}"
            elif tipo == "not_null":
                nulos = df.filter(F.col(coluna).isNull()).count()
                ok, detalhe = nulos == 0, f"{nulos} nulos"
            elif tipo == "unique":
                cols = coluna if isinstance(coluna, list) else [coluna]
                dups = df.count() - df.select(*cols).distinct().count()
                ok, detalhe = dups == 0, f"{dups} duplicatas (chave={cols})"
            elif tipo == "range":
                mn, mx = valor
                fora = df.filter((F.col(coluna) < mn) | (F.col(coluna) > mx)).count()
                ok, detalhe = fora == 0, f"{fora} fora de [{mn},{mx}]"
        except Exception as e:
            ok, detalhe = False, f"Erro: {e}"

        status = "PASS" if ok else ("FAIL" if critico else "WARN")
        print(f"[DQ:SILVER] {status:4} | {tipo:9} | {coluna if coluna else '-'} | {detalhe}")
        if ok: passou += 1
        else:
            falhou += 1
            criticos += 1 if critico else 0

    score = round(passou / len(checks) * 100, 1)
    print(f"[DQ:SILVER] {entidade} | Score={score}% | PASS={passou} FAIL={falhou}\n")
    if criticos > 0:
        raise Exception(f"[DQ:SILVER] {entidade}: {criticos} check(s) critico(s) falharam. Pipeline interrompido.")
    return score

# ============================================================
# REGRAS DE QUALIDADE (uma lista por tabela — como o CHECKS do .py)
# ============================================================
CHECKS = {
    "meta_alfabetizacao_brasil": [
        {"tipo":"min_count","valor":1,                                       "critico":True},
        {"tipo":"not_null", "coluna":"ano",                                  "critico":True},
        {"tipo":"not_null", "coluna":"rede",                                 "critico":True},
        {"tipo":"unique",   "coluna":["ano","rede"],                         "critico":True},
        {"tipo":"range",    "coluna":"taxa_alfabetizacao","valor":(0,100),   "critico":False},
    ],
    "meta_alfabetizacao_uf": [
        {"tipo":"min_count","valor":1,                                       "critico":True},
        {"tipo":"not_null", "coluna":"ano",                                  "critico":True},
        {"tipo":"not_null", "coluna":"sigla_uf",                             "critico":True},
        {"tipo":"not_null", "coluna":"rede",                                 "critico":True},
        {"tipo":"unique",   "coluna":["ano","sigla_uf","rede"],              "critico":True},
        {"tipo":"range",    "coluna":"taxa_alfabetizacao","valor":(0,100),   "critico":False},
    ],
    "meta_alfabetizacao_municipio": [
        {"tipo":"min_count","valor":1,                                       "critico":True},
        {"tipo":"not_null", "coluna":"ano",                                  "critico":True},
        {"tipo":"not_null", "coluna":"id_municipio",                         "critico":True},
        {"tipo":"not_null", "coluna":"rede",                                 "critico":True},
        {"tipo":"unique",   "coluna":["ano","id_municipio","rede"],          "critico":True},
        {"tipo":"range",    "coluna":"taxa_alfabetizacao","valor":(0,100),   "critico":False},
    ],
    "uf": [
        {"tipo":"min_count","valor":1,                                       "critico":True},
        {"tipo":"not_null", "coluna":"ano",                                  "critico":True},
        {"tipo":"not_null", "coluna":"id_uf",                                "critico":True},
        {"tipo":"not_null", "coluna":"rede",                                 "critico":True},
        {"tipo":"unique",   "coluna":["ano","id_uf","rede"],                 "critico":True},
        {"tipo":"range",    "coluna":"taxa_alfabetizacao","valor":(0,100),   "critico":False},
        {"tipo":"range",    "coluna":"media_portugues","valor":(0,1000),     "critico":False},
    ],
    "municipio": [
        {"tipo":"min_count","valor":1,                                       "critico":True},
        {"tipo":"not_null", "coluna":"ano",                                  "critico":True},
        {"tipo":"not_null", "coluna":"id_municipio",                         "critico":True},
        {"tipo":"not_null", "coluna":"rede",                                 "critico":True},
        {"tipo":"unique",   "coluna":["ano","id_municipio","rede"],          "critico":True},
        {"tipo":"range",    "coluna":"taxa_alfabetizacao","valor":(0,100),   "critico":False},
        {"tipo":"range",    "coluna":"media_portugues","valor":(0,1000),     "critico":False},
    ],
    "alunos": [
        {"tipo":"min_count","valor":1,                                       "critico":True},
        {"tipo":"not_null", "coluna":"ano",                                  "critico":True},
        {"tipo":"not_null", "coluna":"id_aluno",                             "critico":True},
        {"tipo":"unique",   "coluna":["ano","id_aluno"],                     "critico":True},
        {"tipo":"range",    "coluna":"proficiencia","valor":(0,1000),        "critico":False},
    ],
}

# ============================================================
# EXECUÇÃO — valida as 6 tabelas Silver
# ============================================================
for tabela, checks in CHECKS.items():
    df = spark.read.parquet(f"{SILVER_BASE}/{tabela}")
    checar_qualidade(tabela, df, checks)

print("Camada Silver validada com sucesso.")

[DQ:SILVER] meta_alfabetizacao_brasil | iniciando | checks=5


[DQ:SILVER] PASS | min_count | - | contagem=3 | minimo=1
[DQ:SILVER] PASS | not_null  | ano | 0 nulos


[DQ:SILVER] PASS | not_null  | rede | 0 nulos


[DQ:SILVER] PASS | unique    | ['ano', 'rede'] | 0 duplicatas (chave=['ano', 'rede'])


[DQ:SILVER] PASS | range     | taxa_alfabetizacao | 0 fora de [0,100]
[DQ:SILVER] meta_alfabetizacao_brasil | Score=100.0% | PASS=5 FAIL=0

[DQ:SILVER] meta_alfabetizacao_uf | iniciando | checks=6


...

[DQ:SILVER] PASS | min_count | - | contagem=6090791 | minimo=1
[DQ:SILVER] PASS | not_null  | ano | 0 nulos


[DQ:SILVER] PASS | not_null  | id_aluno | 0 nulos


26/07/08 17:42:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 17:42:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 17:42:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 17:42:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 17:42:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 17:42:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 17:42:47 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 17:42:47 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 17:42:52 WARN RowBasedKeyValueBatch: Calling spill() on

[DQ:SILVER] PASS | unique    | ['ano', 'id_aluno'] | 0 duplicatas (chave=['ano', 'id_aluno'])


[DQ:SILVER] PASS | range     | proficiencia | 0 fora de [0,1000]
[DQ:SILVER] alunos | Score=100.0% | PASS=5 FAIL=0

Camada Silver validada com sucesso.
